# Data cleaning

In [1]:
import pandas as pd
from collections import Counter
import numpy as np

In [2]:
# paths
from ipcei_spillovers.io import data_path, results_path, load_df, save_df

#### 1. Inspect raw data

In [9]:
# -----------------------------
# Configuration
# -----------------------------
LINKEDIN_FILES = {
    "firms": "Linkedin_firms_jan26.csv",
    "meta": "Linkedin_dynamic_meta_jan26.csv",
    "locations": "Linkedin_locations_jan26.csv",
    "partnerships": "Network_partnerships_jan26.csv",
    "collaborations": "Linkedin_collaborations_jan26.csv",
}

MISSING_TOKENS = {"", "nan", "none", "null", "n/a", "na", "missing", "unknown"}

# -----------------------------
# Audit function
# -----------------------------
def missing_token_audit(df: pd.DataFrame, table_name: str) -> pd.DataFrame:
    rows = []

    for col in df.columns:
        if not pd.api.types.is_object_dtype(df[col]):
            continue

        s = df[col].astype("string")
        stripped = s.str.strip()
        lowered = stripped.str.lower()

        token_counts = Counter(lowered.dropna()[lowered.dropna().isin(MISSING_TOKENS)])

        suspicious_long_text = (
            stripped
            .str.contains(r"\b(nan|none|null|n/a)\b", case=False, na=False)
            & (stripped.str.len() > 40)
        ).sum()

        rows.append({
            "table": table_name,
            "column": col,
            "rows": len(df),
            "true_na": int(s.isna().sum()),
            "blank_strings": int((stripped == "").sum()),
            "explicit_missing_tokens": sum(token_counts.values()),
            "distinct_missing_tokens": ", ".join(sorted(token_counts.keys())),
            "suspicious_long_missing_text": int(suspicious_long_text),
        })

    return pd.DataFrame(rows)

# -----------------------------
# Run audit
# -----------------------------
audits = []

for name, fname in LINKEDIN_FILES.items():
    df = load_df("raw", "linkedin", fname, encoding="utf-8-sig", low_memory=False)
    audits.append(missing_token_audit(df, name))

missing_report = pd.concat(audits, ignore_index=True)

# Sort to see the worst columns first
missing_report = missing_report.sort_values(
    ["suspicious_long_missing_text", "explicit_missing_tokens"],
    ascending=False
)

missing_report

/var/folders/1w/t9ryrq_57m77f4wzbw78745m044mx9/T/ipykernel_61886/2661300879.py:31: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  stripped
/var/folders/1w/t9ryrq_57m77f4wzbw78745m044mx9/T/ipykernel_61886/2661300879.py:31: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  stripped
/var/folders/1w/t9ryrq_57m77f4wzbw78745m044mx9/T/ipykernel_61886/2661300879.py:31: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  stripped
/var/folders/1w/t9ryrq_57m77f4wzbw78745m044mx9/T/ipykernel_61886/2661300879.py:31: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  stripped
/var/folders/1w/t9ryrq_57m77f4wzbw78745m044mx9/T/ipykernel_61886/2661300879.py:31: UserWarni

,table,column,rows,true_na,blank_strings,explicit_missing_tokens,distinct_missing_tokens,suspicious_long_missing_text
14,firms,firm_type,276351,93707,0,0,,2978
9,firms,description,276351,96786,10,12,", n/a",273
15,firms,sector,276351,93707,0,0,,190
28,locations,street,437172,81518,0,3,"n/a, na",27
21,meta,specialities,828233,0,0,0,,24
38,collaborations,collaborating_organizations,769167,0,0,0,,11
8,firms,linkedinUrl,276351,93493,0,0,,2
27,locations,postalCode,437172,102619,8,33,", n/a, na, none",0
26,locations,city,437172,28203,1,1,,0
0,firms,Linkedin_name,276351,86870,0,0,,0


#### 2. Inspect variables of interest more closely

In [12]:
version = "jan26"

# -----------------------
# Files
# -----------------------
FILES = {
    "firms": f"Linkedin_firms_{version}.csv",
    "meta": f"Linkedin_dynamic_meta_{version}.csv",
    "locations": f"Linkedin_locations_{version}.csv",
}

# -----------------------
# Variables you keep in processing_linkedin.py
# -----------------------
KEEP = {
    "firms": [
        "Linkedin_name", "foundingYear", "website", "name", "linkedinUrl",
        "NEW_Linkedin_name", "firm_type", "sector", "industry", "meta_update",
    ],
    "meta": [
        "Linkedin_name", "staffCount", "staffCountRange",
    ],
    "locations": [
        "Linkedin_name", "country", "city", "headquarter", "postalCode",
        "latitude", "longitude", "closingDate", "location_closed",
    ],
}

# -----------------------
# Missing encodings to detect
# -----------------------
MISSING_TOKENS = {"", "nan", "none", "null", "n/a", "na", "missing", "unknown", "-", "--"}
MISSING_REGEX = r"\b(?:nan|none|null|n/a|na)\b"   # non-capturing group


def missing_breakdown(series: pd.Series) -> dict:
    """Return counts by missing type for one column."""
    s = series
    out = {}
    out["rows"] = len(s)
    out["true_na"] = int(pd.isna(s).sum())

    s_str = s.astype("string")
    stripped = s_str.str.strip()
    lowered = stripped.str.lower()

    out["blank_strings"] = int((stripped == "").sum())

    # literal tokens (exact match)
    token_series = lowered.dropna()
    out["explicit_missing_tokens"] = int(token_series.isin(MISSING_TOKENS).sum())

    # suspicious long text containing missing tokens (LLM-ish, "nan means ...")
    mask_susp = stripped.str.contains(MISSING_REGEX, case=False, na=False) & (stripped.str.len() > 40)
    out["suspicious_long_missing_text"] = int(mask_susp.sum())

    # total "missing-like" (not mutually exclusive, but useful as upper bound)
    out["any_missing_like"] = int(
        out["true_na"]
        + out["blank_strings"]
        + out["explicit_missing_tokens"]
        + out["suspicious_long_missing_text"]
    )

    return out


def value_inventory(series: pd.Series, top_n=30, tail_n=20, show_na=True) -> pd.DataFrame:
    """
    Frequency table for a column:
    - top_n most common values
    - tail_n least common values (outliers)
    """
    s = series.astype("string").str.strip()

    vc = s.value_counts(dropna=not show_na)
    top = vc.head(top_n).rename("count").to_frame()
    tail = vc.tail(tail_n).rename("count").to_frame()

    top.index = top.index.fillna("<NA>")
    tail.index = tail.index.fillna("<NA>")

    top = top.reset_index().rename(columns={"index": "value"})
    tail = tail.reset_index().rename(columns={"index": "value"})

    top["rank_bucket"] = "TOP"
    tail["rank_bucket"] = "TAIL"

    return pd.concat([top, tail], ignore_index=True)


# -----------------------
# Load + subset + analyze
# -----------------------
tables = {}
overview_rows = []
inventories = {}  # (table, col) -> df

for table_name, fname in FILES.items():
    df = load_df("raw", "linkedin", fname, encoding="utf-8-sig", low_memory=False)

    keep_cols = [c for c in KEEP[table_name] if c in df.columns]
    df = df[keep_cols].copy()
    tables[table_name] = df

    # Overview by column
    for col in df.columns:
        stats = missing_breakdown(df[col])
        stats.update({"table": table_name, "column": col, "dtype": str(df[col].dtype)})
        overview_rows.append(stats)

        # Inventory (only meaningful for non-numeric too, but works everywhere)
        inventories[(table_name, col)] = value_inventory(df[col], top_n=25, tail_n=15, show_na=True)

overview = pd.DataFrame(overview_rows).sort_values(
    ["suspicious_long_missing_text", "explicit_missing_tokens", "true_na", "blank_strings"],
    ascending=False
)

overview


,rows,true_na,blank_strings,explicit_missing_tokens,suspicious_long_missing_text,any_missing_like,table,column,dtype
6,276351,93707,0,1,2978,96686,firms,firm_type,object
7,276351,93707,0,0,190,93897,firms,sector,object
4,276351,93493,0,0,34,93527,firms,linkedinUrl,object
2,276351,111290,0,0,11,111301,firms,website,object
3,276351,6621,0,0,10,6631,firms,name,object
10,828233,0,0,0,5,5,meta,Linkedin_name,object
0,276351,86870,0,0,1,86871,firms,Linkedin_name,object
13,437172,1,0,0,1,2,locations,Linkedin_name,object
17,437172,102619,8,278,0,102905,locations,postalCode,object
15,437172,28203,1,31,0,28235,locations,city,object


In [19]:
inventories[("firms", "firm_type")]
#inventories[("locations", "country")]

,linkedinUrl,count,rank_bucket
0,<NA>,93493,TOP
1,https://www.linkedin.com/company/goldman-sachs,33,TOP
2,https://www.linkedin.com/company/bp,27,TOP
3,https://www.linkedin.com/company/gsk,25,TOP
4,https://www.linkedin.com/company/dominionenergy,23,TOP
5,https://www.linkedin.com/company/rio-tinto,20,TOP
6,https://www.linkedin.com/company/blackrock,19,TOP
7,https://www.linkedin.com/company/aes,18,TOP
8,https://www.linkedin.com/company/invesco-ltd,18,TOP
9,https://www.linkedin.com/company/dow-chemical,16,TOP


#### 3. Inspect problematic cases

In [20]:
def sector_artifact_report(df: pd.DataFrame, col="sector", top_n=20):
    s = df[col].astype("string").str.strip()

    # Heuristic flags for "LLM explanation" artifacts
    has_newlines = s.str.contains(r"\n", na=False)
    very_long = s.str.len().fillna(0) > 120
    mentions_expl = s.str.contains(r"\b(note:|therefore|classified|does not fit|not explicitly|description)\b",
                                   case=False, na=False)

    starts_other = s.str.startswith("Other Sectors", na=False)

    artifact = (starts_other & (has_newlines | very_long | mentions_expl))

    report = pd.DataFrame({
        "rows": [len(df)],
        "non_missing": [int(s.notna().sum())],
        "unique": [int(s.nunique(dropna=True))],
        "artifact_rows": [int(artifact.sum())],
        "artifact_unique_values": [int(s[artifact].nunique(dropna=True))],
    })

    examples = (
        s[artifact]
        .value_counts()
        .head(top_n)
        .rename("count")
        .to_frame()
        .reset_index()
        .rename(columns={"index": col})
    )

    return report, examples, df.loc[artifact, ["Linkedin_name", col]].head(10)

# usage (firms table)
sector_report, sector_examples, sector_preview = sector_artifact_report(tables["firms"])
sector_report, sector_examples, sector_preview


/var/folders/1w/t9ryrq_57m77f4wzbw78745m044mx9/T/ipykernel_61886/3646060397.py:7: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mentions_expl = s.str.contains(r"\b(note:|therefore|classified|does not fit|not explicitly|description)\b",


(     rows  non_missing  unique  artifact_rows  artifact_unique_values
 0  276351       182644   19336           3444                    3439,
                                                sector  count
 0   Other Sectors\n\nPlease, let me know if I can ...      3
 1   Other Sectors\n\nThe Department for Environmen...      2
 2   Other Sectors\n\nPlease let me know if there's...      2
 3   Other Sectors\n\n(Text: iGEM; The Internationa...      2
 4   Other Sectors\n\nThe description of San Jose S...      1
 5   Other Sectors\n\n(While City drainage and wast...      1
 6   Other Sectors\n(For "Text: nan" it seems there...      1
 7   Other Sectors\n\nGiven that the University of ...      1
 8   Other Sectors\nThe description provided for So...      1
 9   Other Sectors\n\nThe organization, BII, focuse...      1
 10  Other Sectors\n\nThe Government College Univer...      1
 11  Other Sectors\n\nThe description of Æquis Grou...      1
 12  Other Sectors\n\nThe provided text for the Int

In [21]:
def collapse_other_sectors_artifacts(series: pd.Series) -> pd.Series:
    s = series.astype("string").str.strip()

    artifact = (
        s.str.startswith("Other Sectors", na=False)
        & (
            s.str.contains(r"\n", na=False)
            | (s.str.len().fillna(0) > 120)
            | s.str.contains(r"\b(note:|therefore|classified|does not fit|not explicitly|description)\b",
                             case=False, na=False)
        )
    )

    out = s.copy()
    out.loc[artifact] = "Other Sectors"
    return out

# test impact
firms = tables["firms"].copy()
before_unique = firms["sector"].astype("string").nunique(dropna=True)
firms["sector_clean_preview"] = collapse_other_sectors_artifacts(firms["sector"])
after_unique = firms["sector_clean_preview"].nunique(dropna=True)

before_unique, after_unique, firms["sector_clean_preview"].value_counts(dropna=False).head(10)


/var/folders/1w/t9ryrq_57m77f4wzbw78745m044mx9/T/ipykernel_61886/3760191725.py:9: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  | s.str.contains(r"\b(note:|therefore|classified|does not fit|not explicitly|description)\b",


(19724,
 15897,
 sector_clean_preview
 <NA>                   93707
 Other Sectors          74054
 Other Construction      5965
 Other Manufacturing     5248
 Solar                   4907
 Other ICT               3892
 Automotive              3678
 Other Transport         3438
 Built Environment       2822
 Agriculture             2574
 Name: count, dtype: Int64)

In [22]:
def country_code_audit(df: pd.DataFrame, col="country"):
    s = df[col].astype("string").str.strip().str.upper()

    # basic format: two letters
    two_letter = s.str.fullmatch(r"[A-Z]{2}", na=False)

    # Try ISO validation via pycountry (if installed)
    iso_valid = pd.Series([pd.NA] * len(s), index=s.index, dtype="boolean")
    try:
        import pycountry
        valid_set = {c.alpha_2 for c in pycountry.countries}
        iso_valid = s.isin(valid_set)
    except Exception:
        # fallback: we can still inspect unusual codes
        pass

    summary = pd.DataFrame({
        "rows": [len(df)],
        "non_missing": [int(s.notna().sum())],
        "two_letter_format": [int(two_letter.sum())],
        "iso_valid_true": [int(iso_valid.fillna(False).sum())],
        "iso_valid_false": [int((iso_valid == False).sum())],
        "iso_valid_na": [int(iso_valid.isna().sum())],
    })

    unknown_values = (
        s[~s.isna() & two_letter & (iso_valid == False)]
        .value_counts()
        .rename("count")
        .to_frame()
        .reset_index()
        .rename(columns={"index": col})
    )

    weird_non_two_letter = (
        s[~s.isna() & ~two_letter]
        .value_counts()
        .rename("count")
        .to_frame()
        .reset_index()
        .rename(columns={"index": col})
    )

    return summary, unknown_values, weird_non_two_letter

# usage (locations table)
country_summary, unknown_codes, weird_country_values = country_code_audit(tables["locations"])
country_summary, unknown_codes.head(50), weird_country_values.head(50)


(     rows  non_missing  two_letter_format  iso_valid_true  iso_valid_false  \
 0  437172       434798             434797               0                0   
 
    iso_valid_na  
 0        437172  ,
 Empty DataFrame
 Columns: [country, count]
 Index: [],
       country  count
 0  ÖSTERREICH      1)

In [23]:
COUNTRY_FIXES = {
    "ÖSTERREICH": "AT",
    "√ÑSTERREICH": "AT",   # your observed mojibake variant
    "TP": "TL",            # old East Timor code -> modern ISO
    # You can decide what to do with non-ISO placeholders:
    "UN": pd.NA,
    "WO": pd.NA,
    "XJ": pd.NA,
}

loc = tables["locations"].copy()
loc["country_clean_preview"] = (
    loc["country"].astype("string").str.strip().str.upper().replace(COUNTRY_FIXES)
)

loc[["country", "country_clean_preview"]].value_counts().head(20)

country  country_clean_preview
US       US                       92531
GB       GB                       39268
DE       DE                       31708
FR       FR                       28630
NL       NL                       18318
AU       AU                       16243
CA       CA                       15612
ES       ES                       15265
IN       IN                       14279
IT       IT                       12354
SE       SE                        8554
BR       BR                        8443
CH       CH                        7948
BE       BE                        7885
CN       CN                        7407
NO       NO                        6571
DK       DK                        6074
PL       PL                        5577
SG       SG                        4867
AT       AT                        4158
Name: count, dtype: int64